# 01 - Preprocessing (MedQuAD)

Runs locally (no GPU needed): install packages, load MedQuAD directly from the HF Hub, tokenize, and verify. No training here.

In [1]:
%pip install -q datasets transformers sentencepiece pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys

cwd = os.getcwd()
REPO_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd

sys.path.append(REPO_ROOT)
os.chdir(REPO_ROOT)

## Load dataset

`preprocess.run` calls `datasets.load_dataset` directly against `keivalya/MedQuad-MedicalQnADataset` on the HF Hub, cleans/dedupes it, formats it as instruction/input/output records, splits it, and saves JSON.

In [3]:
from src.data.preprocess import run

run(output_dir="data")

/Users/satyamyadav/Projects/PEFT/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


train=13089 val=1635 test=1635


## Verify

In [4]:
import json

with open("data/train.json") as f:
    train = json.load(f)
with open("data/val.json") as f:
    val = json.load(f)
with open("data/test.json") as f:
    test = json.load(f)

print(f"train={len(train)} val={len(val)} test={len(test)}")
print(json.dumps(train[0], indent=2))

train=13089 val=1635 test=1635
{
  "instruction": "what research (or clinical trials) is being done for Oropharyngeal Cancer ?",
  "input": "",
  "output": "New types of treatment are being tested in clinical trials. Information about clinical trials is available from the NCI website. Patients may want to think about taking part in a clinical trial. For some patients, taking part in a clinical trial may be the best treatment choice. Clinical trials are part of the cancer research process. Clinical trials are done to find out if new cancer treatments are safe and effective or better than the standard treatment. Many of today's standard treatments for cancer are based on earlier clinical trials. Patients who take part in a clinical trial may receive the standard treatment or be among the first to receive a new treatment. Patients who take part in clinical trials also help improve the way cancer will be treated in the future. Even when clinical trials do not lead to effective new treatmen

## Tokenize

Use the base model's tokenizer (from `configs/model.yaml`) to check sequence-length distribution and pick `max_seq_length` for training.

In [5]:
import yaml
from transformers import AutoTokenizer

with open("configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)

tokenizer = AutoTokenizer.from_pretrained(model_cfg["base_model"])


def to_text(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"


lengths = sorted(len(tokenizer(to_text(ex))["input_ids"]) for ex in train)
n = len(lengths)
print(f"min={lengths[0]} mean={sum(lengths) // n} p50={lengths[n // 2]} p95={lengths[int(n * 0.95)]} max={lengths[-1]}")
print(f"configured max_seq_length={model_cfg['max_seq_length']}")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


min=16 mean=286 p50=201 p95=751 max=5450
configured max_seq_length=512


In [6]:
sample = to_text(train[0])
encoded = tokenizer(sample)["input_ids"]
decoded = tokenizer.decode(encoded)
print(decoded[:400])

### Instruction:
what research (or clinical trials) is being done for Oropharyngeal Cancer ?

### Response:
New types of treatment are being tested in clinical trials. Information about clinical trials is available from the NCI website. Patients may want to think about taking part in a clinical trial. For some patients, taking part in a clinical trial may be the best treatment choice. Clinical tri
